# "Flux Already Knows" → Modular Diffusers — smoke (training-free subject-driven generation)

Put a reference subject into a new scene — **no training, no LoRA, no extra weights** (LatentUnfold, arXiv:2504.11478).
Publish PRIVATE `remyxai/flux-subject-flux-modular` → load via `trust_remote_code` → assert `FluxSubjectBlock` → a small run → **spikes**: the mosaic-encode round-trip, the `subject_strength=0` no-op, and the `(1,1)` stock-FLUX path.
Upload `block.py` first. Runtime: A100 · `HUGGINGFACE_TOKEN` · accept [FLUX.1-dev](https://huggingface.co/black-forest-labs/FLUX.1-dev).


## 1 · Install + GPU + auth


In [ ]:
!pip install -q "git+https://github.com/huggingface/diffusers.git" transformers accelerate sentencepiece protobuf


In [ ]:
import torch
assert torch.cuda.is_available(); print("GPU:", torch.cuda.get_device_name(0))
from huggingface_hub import login
try:
    from google.colab import userdata; login(userdata.get("HUGGINGFACE_TOKEN"))
except Exception:
    login()
DEV, DT = "cuda", torch.bfloat16


## 2 · Publish PRIVATE (upload block.py first)


In [ ]:
import os, json
from huggingface_hub import HfApi
assert os.path.exists("block.py"), "Upload block.py first."
open("modular_config.json","w").write(json.dumps({"_class_name":"FluxSubjectBlock","_diffusers_version":"0.41.0.dev0","auto_map":{"ModularPipelineBlocks":"block.FluxSubjectBlock"}},indent=2))
F="black-forest-labs/FLUX.1-dev"
def c(s,l,cl): return [None,None,{"pretrained_model_name_or_path":F,"revision":None,"subfolder":s,"type_hint":[l,cl],"variant":None}]
open("modular_model_index.json","w").write(json.dumps({"_blocks_class_name":"FluxSubjectBlock","_class_name":"ModularPipeline","_diffusers_version":"0.41.0.dev0",
 "text_encoder":c("text_encoder","transformers","CLIPTextModel"),"tokenizer":c("tokenizer","transformers","CLIPTokenizer"),
 "text_encoder_2":c("text_encoder_2","transformers","T5EncoderModel"),"tokenizer_2":c("tokenizer_2","transformers","T5TokenizerFast"),
 "transformer":c("transformer","diffusers","FluxTransformer2DModel"),"vae":c("vae","diffusers","AutoencoderKL"),
 "scheduler":c("scheduler","diffusers","FlowMatchEulerDiscreteScheduler")},indent=2))
api=HfApi(); REPO="remyxai/flux-subject-flux-modular"; api.create_repo(REPO,private=True,repo_type="model",exist_ok=True)
for f in ["block.py","modular_config.json","modular_model_index.json"]: api.upload_file(path_or_fileobj=f,path_in_repo=f,repo_id=REPO)
print("published PRIVATE:", api.list_repo_files(REPO))


## 3 · Load + a reference subject


In [ ]:
from diffusers import ModularPipeline
from PIL import Image, ImageDraw
from io import BytesIO
import requests
from IPython.display import display
pipe = ModularPipeline.from_pretrained("remyxai/flux-subject-flux-modular", trust_remote_code=True)
assert type(pipe.blocks).__name__ == "FluxSubjectBlock", type(pipe.blocks).__name__
print("loaded block:", type(pipe.blocks).__name__)   # expect FluxSubjectBlock
pipe.load_components(dtype=DT); pipe.to(DEV)

IMG_URL = "https://raw.githubusercontent.com/bytedance/LatentUnfold/main/assets/clock1.jpg"  #@param {type:"string"}
try:
    subj = Image.open(BytesIO(requests.get(IMG_URL, timeout=30).content)).convert("RGB")
except Exception as e:
    print("fetch failed, upload one:", e)
    from google.colab import files; up=files.upload(); subj = Image.open(list(up.keys())[0]).convert("RGB")
subj.save("subject.png")
print("reference subject:"); display(subj.resize((320,320)))


## 4 · Milestone A — smoke (a small run)


In [ ]:
import torch
SCENE = "On a wooden desk in a cozy study, next to a stack of books and a cup of coffee."  #@param {type:"string"}
SUBJ  = "bright yellow retro alarm clock"  #@param {type:"string"}
g = torch.Generator(DEV).manual_seed(0)
sm = pipe(subject_image="subject.png", prompt=SCENE, subject_prompt=SUBJ,
          height=384, width=384, num_inference_steps=12, generator=g).images[0]
assert sm.size == (384, 384), sm.size
print("[SMOKE] ran; output size", sm.size)
sm.save("smoke.png"); display(sm)


## 5 · Milestone B — spikes

1. **Mosaic-encode round-trip.** The reference tiles are VAE latents that get decoded at the end, so the
   normalisation has to be an inverse. This checks `[VERIFY-encode]` — the one deliberate deviation
   from the reference — before it can muddy the e2e.
2. **`subject_strength=0` → pure mosaic** (no cascade attention). Still subject-driven; the seam is disarmed.
3. **`grid_shape=(1,1)` → stock FLUX.** No reference tiles at all — must run as plain text-to-image.


In [ ]:
import torch
from PIL import Image
import numpy as np
vae, vsf = pipe.vae, 8
tile = pipe._prepare_subject("subject.png", (384,384), False, DEV)
from diffusers.image_processor import VaeImageProcessor
ip = VaeImageProcessor(vae_scale_factor=vsf)
px = ip.preprocess(tile, height=384, width=384).to(device=DEV, dtype=vae.dtype)
z = vae.encode(px).latent_dist.mode()
zn = (z - vae.config.shift_factor) * vae.config.scaling_factor        # the block's tile encoding
back = vae.decode((zn / vae.config.scaling_factor) + vae.config.shift_factor, return_dict=False)[0]
rt = np.asarray(ip.postprocess(back, output_type="pil")[0]).astype(np.float32)
orig = np.asarray(tile).astype(np.float32)
print(f"[SPIKE 1] tile VAE round-trip L1 = {np.abs(rt-orig).mean():.3f} (low = encode/decode are inverses)")

g = torch.Generator(DEV).manual_seed(0)
z0 = pipe(subject_image="subject.png", prompt=SCENE, subject_prompt=SUBJ, subject_strength=0.0,
          height=384, width=384, num_inference_steps=12, generator=g).images[0]
print("[SPIKE 2] subject_strength=0 (cascade disarmed) ran; output size", z0.size)

g = torch.Generator(DEV).manual_seed(0)
z1 = pipe(subject_image="subject.png", prompt="a red panda, flat vector sticker art",
          grid_shape=(1,1), height=384, width=384, num_inference_steps=12, generator=g).images[0]
print("[SPIKE 3] grid_shape=(1,1) (stock FLUX t2i) ran; output size", z1.size)


## Verdict
`loaded block: FluxSubjectBlock` + a coherent small generation + a low tile round-trip L1 + both no-op paths running = the modular seam works. Then run `e2e.ipynb` for the subject-fidelity / prompt-adherence sweep.
